In [1]:
# This cell defines parameters the pipeline passes in
# When run manually these defaults are used
# When called from the ForEach activity the pipeline overrides file_name

# WHY PARAMETERS:
# The pipeline ForEach loop calls this notebook with the actual filename
# So the same notebook can process any CSV file — not hardcoded to one name
# This is the metadata-driven pattern 

file_name = "listings_usa.csv"   # overridden by pipeline at runtime
source_region = "USA"

# Full path inside the Lakehouse Files section
file_path = f"Files/incoming/{file_name}"

print(f"Processing: {file_name}")
print(f"Source region: {source_region}")
print(f"Full path: {file_path}")

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 3, Finished, Available, Finished, False)

Processing: listings_usa.csv
Source region: USA
Full path: Files/incoming/listings_usa.csv


In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)
from pyspark.sql.functions import current_timestamp, lit

# WHY EXPLICIT SCHEMA:
# Never use inferSchema=True on production pipelines
# CSV schema inference reads the file twice and often gets types wrong
# Dates become strings, integers become longs, nulls confuse detection
# Explicit schema is faster and predictable

schema = StructType([
    StructField("ListingID",    StringType(),  False),
    StructField("AgentName",    StringType(),  True),
    StructField("PropertyType", StringType(),  True),
    StructField("City",         StringType(),  True),
    StructField("State",        StringType(),  True),
    StructField("Country",      StringType(),  True),
    StructField("Bedrooms",     StringType(),  True),  # read as string — cast later
    StructField("Bathrooms",    StringType(),  True),  # read as string — cast later
    StructField("SqFt",         StringType(),  True),  # read as string — cast later
    StructField("ListingPrice", StringType(),  True),  # read as string — cast later
    StructField("ListDate",     StringType(),  True),  # MM/DD/YYYY — parse below
    StructField("Status",       StringType(),  True),
    StructField("Description",  StringType(),  True),
    StructField("URL",          StringType(),  True),
    StructField("LastModified", StringType(),  True),
])

# WHY ALL STRINGS FIRST:
# Read everything as string then cast precisely
# If you cast during read and one row has a bad value the entire file fails
# Reading as string lets you cast per-column with TRY_CAST — bad rows get null
# not file failure

df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("multiLine", "false") \
    .option("escape", '"') \
    .schema(schema) \
    .load(file_path)

raw_count = df_raw.count()
print(f"Records read from {file_name}: {raw_count}")

# Quick sample to confirm reading correctly
df_raw.select(
    "ListingID", "City", "Country", "ListingPrice", "ListDate", "Status"
).show(5, truncate=False)

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 4, Finished, Available, Finished, False)

Records read from listings_usa.csv: 800
+---------+-------------+-------+------------+----------+------+
|ListingID|City         |Country|ListingPrice|ListDate  |Status|
+---------+-------------+-------+------------+----------+------+
|USA-00001|Vancouver    |Canada |1215000     |05/23/2022|Sold  |
|USA-00002|Los Angeles  |USA    |280000      |08/12/2022|Active|
|USA-00003|Chicago      |USA    |1109000     |08/27/2023|Active|
|USA-00004|San Francisco|USA    |4314000     |12/11/2022|Active|
|USA-00005|San Francisco|USA    |3324000     |07/04/2023|Active|
+---------+-------------+-------+------------+----------+------+
only showing top 5 rows



In [3]:
from pyspark.sql.functions import (
    col, to_date, trim, upper, initcap,
    when, lit, current_timestamp,
    regexp_replace
)

# WHY RENAME COLUMNS:
# The Bronze layer should already use unified column names
# This means the Silver notebook does not need to know which source
# each row came from — it just applies quality rules to standard columns
#
# COLUMN MAPPING — USA CSV → Unified schema:
# ListingID    → ListingID      (same)
# AgentName    → AgentName      (same)
# PropertyType → PropertyType   (same, standardise values in Silver)
# City         → City           (same)
# State        → State          (same)
# Country      → Country        (same — already full name)
# Bedrooms     → Bedrooms       (cast to integer)
# Bathrooms    → Bathrooms      (cast to integer)
# SqFt         → AreaSqFt       (rename + cast to double)
# ListingPrice → ListingPrice   (cast to double)
# ListDate     → ListDate       (parse MM/DD/YYYY → date)
# Status       → Status         (standardise: Under Contract → Other)
# Description  → Description    (same)
# URL          → ListingURL     (rename)
# LastModified → dropped        (not needed in Bronze)

df_renamed = df_raw \
    .withColumnRenamed("SqFt", "AreaSqFt") \
    .withColumnRenamed("URL",  "ListingURL")

print("Columns after rename:")
print(df_renamed.columns)

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 5, Finished, Available, Finished, False)

Columns after rename:
['ListingID', 'AgentName', 'PropertyType', 'City', 'State', 'Country', 'Bedrooms', 'Bathrooms', 'AreaSqFt', 'ListingPrice', 'ListDate', 'Status', 'Description', 'ListingURL', 'LastModified']


In [4]:
# WHY PARSE DATE IN BRONZE:
# Bronze stores raw data but date parsing is safe to do here
# because we are not changing the value — just the data type
# MM/DD/YYYY is the US standard and cannot be misread
# to_date with explicit format is safe and deterministic

df_typed = df_renamed \
    .withColumn("Bedrooms",
        col("Bedrooms").cast("integer")) \
    .withColumn("Bathrooms",
        col("Bathrooms").cast("integer")) \
    .withColumn("AreaSqFt",
        col("AreaSqFt").cast("double")) \
    .withColumn("ListingPrice",
        col("ListingPrice").cast("double")) \
    .withColumn("ListDate",
        to_date(col("ListDate"), "MM/dd/yyyy"))

# Verify date parsing worked
null_dates = df_typed.filter(col("ListDate").isNull()).count()
print(f"Null dates after parsing: {null_dates}  (should be 0)")

df_typed.select("ListingID", "ListDate", "Bedrooms", "ListingPrice") \
    .show(5, truncate=False)

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 6, Finished, Available, Finished, False)

Null dates after parsing: 0  (should be 0)
+---------+----------+--------+------------+
|ListingID|ListDate  |Bedrooms|ListingPrice|
+---------+----------+--------+------------+
|USA-00001|2022-05-23|1       |1215000.0   |
|USA-00002|2022-08-12|5       |280000.0    |
|USA-00003|2023-08-27|6       |1109000.0   |
|USA-00004|2022-12-11|3       |4314000.0   |
|USA-00005|2023-07-04|5       |3324000.0   |
+---------+----------+--------+------------+
only showing top 5 rows



In [5]:
# WHY STANDARDISE STATUS IN BRONZE:
# Bronze should preserve as much of the original data as possible
# but status is a categorical field that must be unified early
# Otherwise the Silver deduplication logic cannot compare statuses
# across sources
#
# USA STATUS MAPPING:
# Active         → Active   (keep)
# Sold           → Sold     (keep)
# Under Contract → Other    (US-specific — no equivalent in EU/APAC)

df_standardised = df_typed \
    .withColumn("Status",
        when(col("Status") == "Active",           lit("Active"))
       .when(col("Status") == "Sold",             lit("Sold"))
       .when(col("Status") == "Under Contract",   lit("Other"))
       .otherwise(lit("Other")))

# Check status distribution
print("Status distribution after standardisation:")
df_standardised.groupBy("Status").count() \
    .orderBy("count", ascending=False).show()

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 7, Finished, Available, Finished, False)

Status distribution after standardisation:
+------+-----+
|Status|count|
+------+-----+
|Active|  485|
|  Sold|  233|
| Other|   82|
+------+-----+



In [6]:
# WHY THESE AUDIT COLUMNS:
# SourceRegion — tells Silver which regional schema this came from
# SourceFile   — exact filename for lineage — if data is wrong you know
#                exactly which file introduced it
# _ingestion_ts — when this record was loaded into Bronze

df_bronze = df_standardised \
    .withColumn("SourceRegion",   lit("USA")) \
    .withColumn("SourceFile",     lit(file_name)) \
    .withColumn("_ingestion_ts",  current_timestamp()) \
    .drop("LastModified")  # drop source-specific column not in unified schema

print(f"Final column count: {len(df_bronze.columns)}")
print("Final columns:")
print(df_bronze.columns)

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 8, Finished, Available, Finished, False)

Final column count: 17
Final columns:
['ListingID', 'AgentName', 'PropertyType', 'City', 'State', 'Country', 'Bedrooms', 'Bathrooms', 'AreaSqFt', 'ListingPrice', 'ListDate', 'Status', 'Description', 'ListingURL', 'SourceRegion', 'SourceFile', '_ingestion_ts']


In [7]:
# WHY APPEND NOT OVERWRITE:
# bronze_listings receives data from 3 different notebooks
# NB_01 writes USA rows, NB_02 writes Europe rows, NB_03 writes APAC rows
# All 3 append to the same table — overwrite would delete the other regions
#
# WHY mergeSchema:
# The APAC file has extra columns (ViewType, FurnishedStatus)
# mergeSchema allows Delta to add new columns without failing
# Extra columns from APAC will be null for USA and Europe rows — correct behaviour

# Check if table exists — create on first run, append on subsequent runs
if not spark.catalog.tableExists("bronze_listings"):
    df_bronze.write \
        .format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze_listings")
    print(f"Created bronze_listings with {df_bronze.count()} USA rows")
else:
    # Remove existing USA rows before appending
    # This makes the notebook idempotent — safe to rerun
    from delta.tables import DeltaTable
    target = DeltaTable.forName(spark, "bronze_listings")
    target.delete("SourceRegion = 'USA'")
    print(f"Removed existing USA rows")

    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze_listings")
    print(f"Appended {df_bronze.count()} USA rows")

# Final count
total = spark.table("bronze_listings").count()
usa_count = spark.sql(
    "SELECT COUNT(*) FROM bronze_listings WHERE SourceRegion = 'USA'"
).collect()[0][0]
print(f"\nbronze_listings total: {total}  |  USA rows: {usa_count}")

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 9, Finished, Available, Finished, False)

Created bronze_listings with 800 USA rows

bronze_listings total: 800  |  USA rows: 800


In [9]:
print("=== NB_01 — USA CSV Verification ===")

# Row count by region
spark.sql("""
    SELECT SourceRegion, COUNT(*) AS Rows
    FROM   bronze_listings
    GROUP BY SourceRegion
""").show()

# Status distribution
spark.sql("""
    SELECT Status, COUNT(*) AS Count
    FROM   bronze_listings
    WHERE  SourceRegion = 'USA'
    GROUP BY Status
    ORDER BY Count DESC
""").show()

# Property type breakdown
spark.sql("""
    SELECT PropertyType, COUNT(*) AS Count
    FROM   bronze_listings
    WHERE  SourceRegion = 'USA'
    GROUP BY PropertyType
    ORDER BY Count DESC
""").show()

# Price range
spark.sql("""
    SELECT
        ROUND(MIN(ListingPrice), 0) AS MinPrice,
        ROUND(MAX(ListingPrice), 0) AS MaxPrice,
        ROUND(AVG(ListingPrice), 0) AS AvgPrice
    FROM bronze_listings
    WHERE SourceRegion = 'USA'
""").show()

# Sample rows
spark.table("bronze_listings") \
    .filter("SourceRegion = 'USA'") \
    .select("ListingID", "City", "Country",
            "Bedrooms", "AreaSqFt", "ListingPrice",
            "ListDate", "Status", "SourceFile") \
    .show(5, truncate=False)

StatementMeta(, 52bd18e8-a653-480f-9c96-6663c2787b42, 11, Finished, Available, Finished, False)

=== NB_01 — USA CSV Verification ===
+------------+----+
|SourceRegion|Rows|
+------------+----+
|         USA| 800|
+------------+----+

+------+-----+
|Status|Count|
+------+-----+
|Active|  485|
|  Sold|  233|
| Other|   82|
+------+-----+

+-------------+-----+
| PropertyType|Count|
+-------------+-----+
|        Condo|  144|
|Single Family|  139|
|    Townhouse|  132|
| Multi-Family|  131|
|    Penthouse|  131|
|        Villa|  123|
+-------------+-----+

+--------+---------+---------+
|MinPrice| MaxPrice| AvgPrice|
+--------+---------+---------+
|151000.0|4498000.0|2326461.0|
+--------+---------+---------+

+---------+-------------+-------+--------+--------+------------+----------+------+----------------+
|ListingID|City         |Country|Bedrooms|AreaSqFt|ListingPrice|ListDate  |Status|SourceFile      |
+---------+-------------+-------+--------+--------+------------+----------+------+----------------+
|USA-00001|Vancouver    |Canada |1       |5006.0  |1215000.0   |2022-05-23|Sold